# KIPS 2026 Item Query NLU experiments

이 노트북은 `experiments/nlu/kips_2026_item_query/`의 실험 코드를 Colab에서 실행하는 launcher입니다.

1. 공개 저장소 clone 및 패키지 설치
2. GPU 런타임 선택
3. 아래 셀을 순서대로 실행
4. Main 9 runs 완료 후 필요할 때만 ablation / R-Drop 실행


## 1. Public repository clone + package install

In [ ]:
import os, subprocess
from pathlib import Path

repo_dir = Path('/content/pumpkin_public')
if not repo_dir.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/hammonsterwow/pumpkin_public.git',
        str(repo_dir),
    ], check=True)

os.chdir(repo_dir)
subprocess.run([
    'pip', 'install', '-q', '-r',
    'experiments/nlu/kips_2026_item_query/requirements-colab.txt',
], check=True)
print('repo:', repo_dir)


## 2. GPU 확인

In [ ]:
import torch
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))


## 3. Main experiment — M0/M1/M2 × seeds 42/43/44 = 9 runs

In [ ]:
subprocess.run([
    'python', '-m', 'experiments.nlu.kips_2026_item_query.run_all',
    '--mode', 'main',
    '--result-dir', '/content/kips_2026_results',
], check=True)


## 4. Main 결과표

In [ ]:
import pandas as pd
paper_table = pd.read_csv('/content/kips_2026_results/paper_main_table.csv')
main_summary = pd.read_csv('/content/kips_2026_results/main_summary.csv')
display(paper_table)
display(main_summary)


## 5. Ablation screening — 선택

`M1 → + differential LR → + oversampling → + weighted loss(M2)` 순으로 하나씩 추가합니다. 처음에는 seed 42 하나만 실행합니다.


In [ ]:
subprocess.run([
    'python', '-m', 'experiments.nlu.kips_2026_item_query.run_all',
    '--mode', 'ablation',
    '--ablation-seeds', '42',
    '--result-dir', '/content/kips_2026_results',
], check=True)

ablation = pd.read_csv('/content/kips_2026_results/ablation_summary.csv')
display(ablation)


## 6. Ablation 3 seeds — 논문에 넣기로 결정했을 때만

In [ ]:
subprocess.run([
    'python', '-m', 'experiments.nlu.kips_2026_item_query.run_all',
    '--mode', 'ablation',
    '--ablation-seeds', '42,43,44',
    '--result-dir', '/content/kips_2026_results',
], check=True)


## 7. Optional R-Drop screening

In [ ]:
subprocess.run([
    'python', '-m', 'experiments.nlu.kips_2026_item_query.run_all',
    '--mode', 'ablation',
    '--ablation-seeds', '42', '--rdrop',
    '--result-dir', '/content/kips_2026_results',
], check=True)


## 8. 결과 zip 다운로드

In [ ]:
from google.colab import files
files.download('/content/kips_2026_results.zip')
